In [9]:
import json
import os
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import (
    RunnablePassthrough,
)
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import AIMessage, HumanMessage

# Langchain Specific imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

load_dotenv()


True

## Data Ingestion and Loading

In [3]:
## create sample documents
sample_docs = [
    Document(page_content="""
        Machine Learning Basics

        Machine Learning (ML) is a subset of Artificial Intelligence that enables systems to learn patterns from data and make predictions or decisions without explicit programming. It revolves around three key components — Task (T), Experience (E), and Performance (P) — where models improve performance on a task through experience over time.

        Core Types of ML include:
        Supervised Learning – Uses labeled data for tasks like classification (predicting categories) and regression (predicting continuous values).
        Unsupervised Learning – Works with unlabeled data to find patterns, such as clustering or dimensionality reduction.
        Reinforcement Learning – Learns optimal actions through trial and error to maximize rewards.
        Additional: Semi-Supervised and Self-Supervised Learning for scenarios with limited labeled data.
        """,
        metadata = {"source": "Machine Learning Basics.txt", "page":1, "section": "Introduction", "keywords": ["machine learning", "supervised learning", "unsupervised learning", "reinforcement learning"]}
    ),
    Document(page_content="""
        Deep Learning Essentials

        Deep learning is a subset of machine learning that uses artificial neural networks with multiple layers to automatically learn complex patterns from large datasets. Inspired by the human brain, these networks consist of input layers, hidden layers, and output layers, where each neuron applies nonlinear transformations to extract features and make predictions,
        Key differences from traditional machine learning include automated feature extraction, end-to-end learning, and the ability to handle unstructured data like images, audio, and text. However, deep learning requires large datasets, high computational power (GPUs/TPUs), and often suffers from interpretability challenges.
        """,
        metadata = {"source": "Deep Learning Essentials.txt", "page":1, "section": "Introduction", "keywords": ["deep learning", "neural networks", "feature extraction"]}
    ),
    Document(page_content="""

        Large Language Models
        Large Language Models (LLMs) are advanced deep learning systems trained on massive datasets—often billions or trillions of words—to understand, generate, and manipulate human-like language. They are built on transformer architectures that use self-attention mechanisms to process sequences in parallel, capturing context and relationships between words even when far apart in text .
        Core Functionality: LLMs work by predicting the next token (word or subword) in a sequence based on context. This enables them to perform tasks like text generation, summarization, translation, code writing, sentiment analysis, and even multimodal outputs such as images or audio when extended into Large Multimodal Models (LMMs)
        """,
        metadata = {"source": "Large Language Models.txt", "page":1, "section": "Introduction", "keywords": ["large language models", "transformers", "natural language processing"]}
    )
]

print(sample_docs)

[Document(metadata={'source': 'Machine Learning Basics.txt', 'page': 1, 'section': 'Introduction', 'keywords': ['machine learning', 'supervised learning', 'unsupervised learning', 'reinforcement learning']}, page_content='\n        Machine Learning Basics\n\n        Machine Learning (ML) is a subset of Artificial Intelligence that enables systems to learn patterns from data and make predictions or decisions without explicit programming. It revolves around three key components — Task (T), Experience (E), and Performance (P) — where models improve performance on a task through experience over time.\n\n        Core Types of ML include:\n        Supervised Learning – Uses labeled data for tasks like classification (predicting categories) and regression (predicting continuous values).\n        Unsupervised Learning – Works with unlabeled data to find patterns, such as clustering or dimensionality reduction.\n        Reinforcement Learning – Learns optimal actions through trial and error to 

### Text Splitting

In [7]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=[" \n\n", "\n", " ", ""]
)

## Split the documents into chunks
chunks = text_splitter.split_documents(sample_docs)
print(chunks[0])
print(chunks[1])

page_content='Machine Learning Basics

        Machine Learning (ML) is a subset of Artificial Intelligence that enables systems to learn patterns from data and make predictions or decisions without explicit programming. It revolves around three key components — Task (T), Experience (E), and Performance (P) — where models improve performance on a task through experience over time.

        Core Types of ML include:' metadata={'source': 'Machine Learning Basics.txt', 'page': 1, 'section': 'Introduction', 'keywords': ['machine learning', 'supervised learning', 'unsupervised learning', 'reinforcement learning']}
page_content='Core Types of ML include:
        Supervised Learning – Uses labeled data for tasks like classification (predicting categories) and regression (predicting continuous values).
        Unsupervised Learning – Works with unlabeled data to find patterns, such as clustering or dimensionality reduction.
        Reinforcement Learning – Learns optimal actions through trial 

## Load the Embedding Models

In [10]:
## Initialize a simple Embedding model (no API Key needed!)
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    # model_name="sentence-transformers/paraphrase-multilingual-MinLM-L12-v2",
    multi_process=True,
    show_progress=True,
    cache_folder="./../model_cache/",
    # model_kwargs = {"device": "cpu"}
    # model_kwargs = {"device": "gpu"}
    # model_kwargs = {"device": "cuda"}
)
embedding_model

HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder='./../model_cache/', model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=True, show_progress=True)

In [11]:
sample_text = "Machine Learning is facinating"
vector = embedding_model.embed_query(sample_text)
vector

[-0.011421240866184235,
 -0.09401240199804306,
 0.07909874618053436,
 0.01005230937153101,
 0.03052467852830887,
 -0.07159081101417542,
 0.025884579867124557,
 -0.04600030556321144,
 -0.010964806191623211,
 -0.004916173405945301,
 -0.08440316468477249,
 0.017160382121801376,
 0.035648174583911896,
 -0.03660526126623154,
 -0.16236746311187744,
 -0.05686250329017639,
 -0.007439363747835159,
 -0.04244787245988846,
 -0.052435532212257385,
 -0.06360574811697006,
 -0.05709030479192734,
 -0.010202315635979176,
 -0.03527823090553284,
 0.016096273437142372,
 0.07030282914638519,
 -0.014181132428348064,
 0.021265951916575432,
 0.01020015124231577,
 0.009754045866429806,
 -0.056863293051719666,
 0.038969170302152634,
 0.03804619237780571,
 -0.0020143098663538694,
 -0.011471729725599289,
 -0.10078305751085281,
 -0.013842337764799595,
 0.042432140558958054,
 0.06592870503664017,
 0.0354066900908947,
 0.005910268519073725,
 0.007874409668147564,
 -0.06986453384160995,
 -0.008599881082773209,
 0.0383

## Create FIASS Vector Store

In [13]:
vector_store = FAISS.from_documents(
    documents=chunks, 
    embedding=embedding_model)

print(f"Vector store created with {vector_store.index.ntotal} vectors.")

Vector store created with 7 vectors.


In [14]:
vector_store

In [15]:
### Save vector store to disk for later use.
vector_store.save_local("faiss_vector_store")
print("Vector store saved to disk.")

Vector store saved to disk.


In [17]:
### load vetor store from disk
loaded_vector_store = FAISS.load_local(
    "faiss_vector_store", 
    embedding_model,
    allow_dangerous_deserialization=True
    )

print(f"Vector store created with {loaded_vector_store.index.ntotal} vectors.")

Vector store created with 7 vectors.


In [19]:
## similarity search on vectors

result_with_scores = vector_store.similarity_search_with_score(query="What is Deep Learning?", k=2)
print("Similarity search with scores:")
for doc, score in result_with_scores:
    print(f"\nScore: {score:.3f}")
    print(f"Document: {doc.page_content[:100]}...")
    print(f"Metadata: {doc.metadata}...")

Similarity search with scores:

Score: 0.523
Document: Deep Learning Essentials

        Deep learning is a subset of machine learning that uses artificial...
Metadata: {'source': 'Deep Learning Essentials.txt', 'page': 1, 'section': 'Introduction', 'keywords': ['deep learning', 'neural networks', 'feature extraction']}...

Score: 0.776
Document: Key differences from traditional machine learning include automated feature extraction, end-to-end l...
Metadata: {'source': 'Deep Learning Essentials.txt', 'page': 1, 'section': 'Introduction', 'keywords': ['deep learning', 'neural networks', 'feature extraction']}...


In [20]:
## Search with Metadata Filtering
filter_dict={"section": "Introduction"}
filtered_results = vector_store.similarity_search(
    query="What is Deep Learning?", 
    k=2, 
    filter=filter_dict
)
print("Similarity search with metadata filtering:")
for doc in filtered_results:
    print(f"\nDocument: {doc.page_content[:100]}...")
    print(f"Metadata: {doc.metadata}...")

Similarity search with metadata filtering:

Document: Deep Learning Essentials

        Deep learning is a subset of machine learning that uses artificial...
Metadata: {'source': 'Deep Learning Essentials.txt', 'page': 1, 'section': 'Introduction', 'keywords': ['deep learning', 'neural networks', 'feature extraction']}...

Document: Key differences from traditional machine learning include automated feature extraction, end-to-end l...
Metadata: {'source': 'Deep Learning Essentials.txt', 'page': 1, 'section': 'Introduction', 'keywords': ['deep learning', 'neural networks', 'feature extraction']}...


In [21]:
len(filtered_results)

2